In [2]:
import json
import requests
import time
import os
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

OPENROUTER_API_KEY = "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"
JSON_FILE_PATH = "health_data.json"  # Path where Wokwi dumps the JSON

def get_ai_response(health_data: dict) -> str:
    prompt = f"""You are a medical assistant. Analyze the following patient health sensor data and provide a brief clinical assessment:

{json.dumps(health_data, indent=2)}

Comment on each vital sign, whether it is normal or abnormal, and summarize the overall health status."""

    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "meta-llama/llama-3-8b-instruct",
            "max_tokens": 300,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    result = response.json()
    return result["choices"][0]["message"]["content"]


def process_json_file(filepath: str):
    try:
        with open(filepath, "r") as f:
            health_data = json.load(f)

        print("\n" + "="*50)
        print("New data received from hardware:")
        print(json.dumps(health_data, indent=2))
        print("="*50)

        print("\nAI Assessment:")
        result = get_ai_response(health_data)
        print(result)
        print("="*50 + "\n")

    except json.JSONDecodeError:
        print("Warning: JSON file is being written, retrying...")
        time.sleep(0.5)
        process_json_file(filepath)  # Retry once file is fully written
    except Exception as e:
        print(f"Error processing file: {e}")


# ---- Watchdog File Handler ----
class HealthDataHandler(FileSystemEventHandler):
    def __init__(self, target_file):
        self.target_file = os.path.abspath(target_file)
        self.last_modified = 0

    def on_modified(self, event):
        # Only react to the specific JSON file
        if os.path.abspath(event.src_path) == self.target_file:
            # Debounce: ignore duplicate events within 1 second
            current_time = time.time()
            if current_time - self.last_modified < 1:
                return
            self.last_modified = current_time

            print(f"\n[DETECTED] {self.target_file} updated by hardware!")
            process_json_file(self.target_file)


# ---- Main ----
if __name__ == "__main__":
    # Run once on startup with existing data
    if os.path.exists(JSON_FILE_PATH):
        print("[STARTUP] Processing existing data...")
        process_json_file(JSON_FILE_PATH)
    else:
        print(f"[WAITING] {JSON_FILE_PATH} not found yet. Waiting for hardware...")

    # Start watching for file changes
    event_handler = HealthDataHandler(JSON_FILE_PATH)
    observer = Observer()
    observer.schedule(event_handler, path=os.path.dirname(os.path.abspath(JSON_FILE_PATH)) or ".", recursive=False)
    observer.start()

    print(f"[WATCHING] Monitoring {JSON_FILE_PATH} for real-time updates...\n")

    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n[STOPPED] Monitoring stopped.")
        observer.stop()

    observer.join()

[WAITING] health_data.json not found yet. Waiting for hardware...
[WATCHING] Monitoring health_data.json for real-time updates...


[STOPPED] Monitoring stopped.
